# web_agent — GOLD dataset run (Kaggle)

Real, leak-safe gold data (before+after images + task only). Separate from the synthetic notebook — the synthetic 70k pipeline is untouched. Heads with no gold labels (confidence/memory/recovery_success) are disabled in `qwen2vl_2b_gold.yaml`. Judge by **outcome_mcc** on the gold TEST split.

In [5]:
# 1. Clone the package + install deps, make it importable (re-run safe)
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
SRC = f"{ROOT}/src"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}
!pip install -q -U "transformers>=4.49" peft bitsandbytes accelerate scikit-learn
for m in [k for k in list(sys.modules) if k == "web_agent" or k.startswith("web_agent.")]:
    del sys.modules[m]
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import web_agent
print("web_agent ready ->", list(web_agent.__path__))

Already up to date.
/kaggle/working/webagent
web_agent ready -> ['/kaggle/working/webagent/src/web_agent']


In [6]:
# 2. GPU + gold data path (auto-detected, slug-proof)
import os, glob
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))
hits = glob.glob("/kaggle/input/**/split_train.json", recursive=True)
assert hits, "WebGoldData not attached: right panel -> Add Input -> WebGoldData"
GOLD_PATH = os.path.dirname(hits[0])
print("GOLD_PATH =", GOLD_PATH)
print("splits:", [f for f in os.listdir(GOLD_PATH) if f.endswith(".json")])

/bin/bash: line 1: nvidia-smi: command not found
input dirs: ['datasets']
GOLD_PATH = /kaggle/input/datasets/kiyasmahmud/webgolddata/web_agent_gold_v8_approved_2032_real_browser
splits: ['split_test.json', 'gold_export_summary.json', 'split_train.json', 'split_val.json', 'blank_filter_summary.json']


In [7]:
# 3. Config (gold) + processor + one gold batch
import torch
from transformers import AutoProcessor
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split

set_seed(42)
cfg = load_config("configs/backbones/qwen2vl_2b_gold.yaml")
cfg["data"]["root"] = GOLD_PATH
cfg["data"]["num_workers"] = 0

bb = cfg["backbone"]
processor = AutoProcessor.from_pretrained(
    bb["vlm_model"], min_pixels=bb["min_pixels"], max_pixels=bb["max_pixels"])

train = load_gold_split(cfg, "train")
print("gold train rows:", len(train))
loader = build_gold_dataloader(cfg, "train", processor, records=train,
                               limit=8, batch_size=4, num_workers=0)
batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:22} {tuple(v.shape)}  {v.dtype}")
print("outcome labels in batch:", batch["label_outcome"].tolist())

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

gold train rows: 1219


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


batch keys: ['bbox', 'bbox_mask', 'borrowed', 'label_outcome', 'label_failtype', 'label_action', 'label_recovery', 'label_memory', 'label_confidence', 'label_recovery_success', 'original_task_id', 'input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'mm_token_type_ids']
  bbox                   (4, 4)  torch.float32
  bbox_mask              (4, 1)  torch.float32
  borrowed               (4, 1)  torch.float32
  label_outcome          (4,)  torch.int64
  label_failtype         (4,)  torch.int64
  label_action           (4,)  torch.int64
  label_recovery         (4,)  torch.int64
  label_memory           (4, 1)  torch.float32
  label_confidence       (4, 1)  torch.float32
  label_recovery_success (4, 1)  torch.float32
  input_ids              (4, 535)  torch.int64
  attention_mask         (4, 535)  torch.int64
  pixel_values           (8064, 1176)  torch.float32
  image_grid_thw         (8, 3)  torch.int64
  mm_token_type_ids      (4, 535)  torch.int64
outcome labels in batch:

In [ ]:
# 4. Build model + gold class-weighted loss (v12: confidence+memory ON, recovery_outcome OFF)
from web_agent.models.model import WebAgentModel
from web_agent.models.loss import CombinedLoss
from web_agent.data.gold_dataloader import gold_class_weights

model = WebAgentModel(cfg)
print("VLM hidden dim D =", model.encoder.hidden_dim, "| pooling =", cfg["backbone"]["pooling"])
device = "cuda"
for m in (model.adapter, model.failure_head, model.action_head,
          model.memory_head, model.recovery_outcome_head):
    m.to(device)

aw, fw, ow = gold_class_weights(train)
print("action w :", [round(x,2) for x in aw.tolist()], "(6 classes incl PRESS_KEY)")
print("failtype w:", [round(x,2) for x in fw.tolist()])
print("outcome w:", [round(x,2) for x in ow.tolist()], "(capped)")
loss_fn = CombinedLoss(cfg, action_class_weights=aw.to(device),
                       failtype_class_weights=fw.to(device),
                       outcome_class_weights=ow.to(device)).to(device)
print("head loss weights -> confidence:", cfg["loss"]["confidence"],
      "| memory:", cfg["loss"]["memory_flag"],
      "| recovery_outcome:", cfg["loss"]["recovery_outcome"], "(0 = off)")
print("trainable params:", f"{sum(p.numel() for p in model.trainable_parameters()):,}")

In [ ]:
# 5. SMOKE — one batch, loss finite, only recovery_outcome disabled
model.train()
b = next(iter(loader))
with torch.autocast("cuda", dtype=torch.float16):
    preds = model(b)
    terms = loss_fn(preds, {k: (v.to(device) if torch.is_tensor(v) else v)
                            for k, v in b.items()})
print("loss terms:", {k: round(float(v.detach()), 4) for k, v in terms.items()})
assert torch.isfinite(terms["total"]), "loss NaN/inf - STOP"
# v12: confidence + memory are trained (real labels); only recovery_outcome is off.
assert float(terms["recovery_outcome"].detach()) * cfg["loss"]["recovery_outcome"] == 0.0, "recovery_outcome not disabled"
print("confidence + memory now trained ->",
      round(float(terms["confidence"].detach()), 4), round(float(terms["memory"].detach()), 4))
print("contrastive non-zero:", float(terms["contrastive"].detach()) != 0.0)
print("peak GPU (GB):", round(torch.cuda.max_memory_allocated()/1e9, 2))
print("SMOKE PASS")

In [ ]:
# 6. TRAIN on gold + final eval on the gold TEST split
from web_agent.train.trainer import Trainer, collect_predictions, compute_metrics

cfg["train"]["epochs"] = 5
cfg["data"]["num_workers"] = 4
train_loader = build_gold_dataloader(cfg, "train", processor, shuffle=True, num_workers=4)
val_loader   = build_gold_dataloader(cfg, "val",   processor, shuffle=False, num_workers=4)

trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader, train_sampler=None)
print("training done:", trainer.fit())

test_loader = build_gold_dataloader(cfg, "test", processor, shuffle=False, num_workers=4)
gold_metrics = compute_metrics(collect_predictions(model, test_loader, device))
print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("  HEADLINE = outcome_mcc / outcome_bal_acc / failure_macro_f1")
print("  v12: memory_acc + ece are now REAL (heads trained).")
print("  ignore only recovery_outcome_acc (that head disabled -- recovery_success too sparse)")

In [ ]:
# 7. Save ALL results tagged with a VERSION name -> download from the Output panel
import os, shutil, json, csv as _csv
assert "gold_metrics" in globals(), "Run cell 6 to completion first (it defines gold_metrics)."
VERSION = "gold_v1"          # <-- change this each run (gold_v1, gold_v2, ...)

outdir = f"/kaggle/working/results_{VERSION}"
os.makedirs(outdir, exist_ok=True)

# final gold-TEST metrics (from cell 6) -> JSON + CSV
with open(f"{outdir}/gold_test_{VERSION}.json", "w") as f:
    json.dump(gold_metrics, f, indent=2)
with open(f"{outdir}/gold_test_{VERSION}.csv", "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["version"] + list(gold_metrics.keys()))
    w.writerow([VERSION] + [round(v, 5) for v in gold_metrics.values()])

# per-epoch metrics CSV written by the Trainer
src_csv = f"/kaggle/working/webagent/{cfg['train']['metrics_csv']}"
if os.path.exists(src_csv):
    shutil.copy(src_csv, f"{outdir}/per_epoch_{VERSION}.csv")

# best checkpoints (top-k by val Failure-F1)
ckdir = f"/kaggle/working/webagent/checkpoints/{cfg['name']}"
if os.path.isdir(ckdir):
    shutil.copytree(ckdir, f"{outdir}/checkpoints", dirs_exist_ok=True)

# one-click zip
zip_path = shutil.make_archive(f"/kaggle/working/results_{VERSION}", "zip", outdir)
print("RESULTS SAVED (CSV + JSON + checkpoints):")
print("  folder:", outdir)
print("  zip   :", zip_path)
print("  csv   :", f"{outdir}/gold_test_{VERSION}.csv")
print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("Download from the right-side Output panel (/kaggle/working).")